```{contents}
```

## Learning Rate Scheduling


The **learning rate (LR)** controls the **step size** of parameter updates during training.
A fixed LR is rarely optimal throughout training because:

* **Early training** benefits from large steps for rapid progress.
* **Later training** requires small steps for fine convergence.

**Learning rate scheduling** dynamically adjusts the LR over time to improve:

* convergence speed,
* training stability,
* final model performance.

$$
\theta_{t+1} = \theta_t - \eta_t \nabla \mathcal{L}(\theta_t)
$$
where $\eta_t$ is the scheduled learning rate at step $t$.

---

### Why Scheduling Matters

| Phase          | Optimal LR Behavior          |
| -------------- | ---------------------------- |
| Early training | Large LR → fast exploration  |
| Mid training   | Moderate LR → stable descent |
| Late training  | Small LR → fine convergence  |

Without scheduling:

* Too large LR → divergence
* Too small LR → slow convergence / poor minima

---

### Major Scheduling Strategies

| Scheduler         | Behavior                     | Typical Use           |
| ----------------- | ---------------------------- | --------------------- |
| Step Decay        | Drop LR at fixed epochs      | Classic CNN training  |
| Exponential Decay | Smooth multiplicative decay  | Long training runs    |
| Cosine Annealing  | Periodic smooth decay        | Vision / Transformers |
| Warmup            | Gradual LR increase at start | Large models          |
| Reduce on Plateau | Decay when validation stalls | General purpose       |
| One-Cycle Policy  | Increase then decrease       | Fast convergence      |

---

### Mathematical Forms

**Step Decay**

$$
\eta_t = \eta_0 \cdot \gamma^{\lfloor t/s \rfloor}
$$

**Exponential Decay**

$$
\eta_t = \eta_0 e^{-kt}
$$

**Cosine Annealing**

$$
\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max}-\eta_{min})(1+\cos(\pi t/T))
$$

---

### Training Workflow with Scheduling

1. Initialize model and optimizer
2. Choose base learning rate
3. Attach scheduler
4. Update LR at each step or epoch
5. Monitor convergence

---

### PyTorch Demonstrations

#### StepLR

```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

for epoch in range(50):
    train()
    scheduler.step()
```

#### ReduceLROnPlateau

```python
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

for epoch in range(50):
    loss = train()
    scheduler.step(loss)
```

#### Cosine Annealing

```python
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
```

#### Warmup + Cosine

```python
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

warmup = LinearLR(optimizer, start_factor=0.1, total_iters=5)
cosine = CosineAnnealingLR(optimizer, T_max=45)

scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[5])
```

#### One Cycle Policy

```python
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.1,
    total_steps=1000
)
```

---

### Practical Guidelines

| Model Type        | Recommended Strategy |
| ----------------- | -------------------- |
| CNNs              | StepLR / Cosine      |
| Transformers      | Warmup + Cosine      |
| Large models      | OneCycle             |
| Unknown landscape | ReduceOnPlateau      |

---

### Variants and Extensions

| Variant          | Purpose                        |
| ---------------- | ------------------------------ |
| Cyclical LR      | Escape sharp minima            |
| Adaptive LR      | Adam / RMSProp                 |
| Layer-wise LR    | Fine-tuning pre-trained models |
| Polynomial decay | Long-horizon training          |

---

### Summary

| Benefit               | Effect             |
| --------------------- | ------------------ |
| Faster convergence    | Early large LR     |
| Better generalization | Late small LR      |
| Higher stability      | Avoid divergence   |
| Automatic adaptation  | Less manual tuning |

---

### Final Insight

> **Learning rate scheduling is the control system that governs how neural networks learn over time.**
